# 05 · Previsões com o modelo vigente e teste integrado

**Etapa do fluxo:** 10 (previsões com o modelo vigente + evidências do ciclo).

O modelo é carregado **a partir do manifesto** `models/modelo_vigente.json`; o arquivo serializado contém o
pré-processamento ajustado e as features eleitas, então a previsão usa exatamente o mesmo tratamento do treino.
As previsões são feitas sobre o **teste reservado**, nunca usado na escolha do modelo.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [2]:
import mlflow
from sklearn.metrics import ConfusionMatrixDisplay
from src.ciclo import carregar_contexto, diretorio_ciclo, obter_cycle_id
from src.modelagem import avaliar, colunas_entrada
from src.promocao import carregar_modelo_vigente
from src.rastreamento import configurar_mlflow, run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
teste = pd.read_csv(caminho(cfg, cfg["dados"]["diretorio_processado"]) / "teste.csv")
alvo, X_cols = cfg["dados"]["alvo"], colunas_entrada(cfg)

modelo, manifesto = carregar_modelo_vigente(cfg)
assert modelo is not None, "Não há modelo vigente para gerar previsões."
colunas_do_modelo = [c for nome, _, cols in modelo.named_steps["preprocessamento"].transformers_ if nome != "remainder" for c in cols]
assert sorted(colunas_do_modelo) == sorted(manifesto["features"]), "Features do manifesto divergem do modelo serializado"
print(json.dumps({k: manifesto[k] for k in ["cycle_id", "data_version", "algoritmo", "run_id", "features", "metrica"]}, indent=2, ensure_ascii=False))

{
  "cycle_id": "c20261123T183614Z-gh45240022843",
  "data_version": "v004-d9278016",
  "algoritmo": "hist_gradient_boosting",
  "run_id": "b5dc8f68a52a44d4952e8171783d6c09",
  "features": [
    "euribor3m",
    "emp.var.rate",
    "cons.price.idx",
    "contatado_antes",
    "contact",
    "nr.employed",
    "month",
    "poutcome",
    "cons.conf.idx",
    "job"
  ],
  "metrica": {
    "nome": "roc_auc",
    "validacao": 0.7401
  }
}


In [3]:
probabilidade = modelo.predict_proba(teste[X_cols])[:, 1]
previsoes = pd.DataFrame({
    "registro_id": teste["registro_id"],
    "probabilidade_yes": probabilidade.round(4),
    "previsto": modelo.predict(teste[X_cols]),
    "real": teste[alvo],
})
previsoes.to_csv(pasta_ciclo / "previsoes.csv", index=False)
metricas_teste = avaliar(modelo, teste[X_cols], teste[alvo])
print("Métricas no teste reservado:", metricas_teste)
display(previsoes.sort_values("probabilidade_yes", ascending=False).head(10))
ConfusionMatrixDisplay.from_predictions(previsoes["real"], previsoes["previsto"], display_labels=["no", "yes"])
plt.title(f"Vigente {manifesto['algoritmo']} - teste")
plt.show()

Métricas no teste reservado: {'f1_macro': 0.6674, 'balanced_accuracy': 0.7574, 'accuracy': 0.8099, 'precision_pos': 0.3337, 'recall_pos': 0.6897, 'n': 4119, 'taxa_positiva_real': 0.1126, 'roc_auc': 0.811}


,registro_id,probabilidade_yes,previsto,real
3643,36283,0.9438,1,0
3844,38313,0.9401,1,1
3859,38453,0.9401,1,1
3857,38432,0.9401,1,1
3827,38155,0.9401,1,1
3900,38798,0.9401,1,1
3918,38968,0.9401,1,1
3888,38723,0.9401,1,1
3895,38759,0.9401,1,1
3927,39042,0.9401,1,1


## Comparação entre ciclos
Histórico persistido em `state/historico_ciclos.csv`: mostra como `data_version`, features eleitas, métricas e decisão
mudaram a cada ciclo.

In [4]:
historico = pd.read_csv(caminho(cfg, cfg["saida"]["historico_ciclos"]))
display(historico)
if len(historico) >= 2:
    anterior, atual = historico.iloc[-2], historico.iloc[-1]
    f_ant, f_atu = set(anterior["features_eleitas"].split("|")), set(atual["features_eleitas"].split("|"))
    print("O que mudou em relação ao ciclo anterior:")
    print("  data_version:", anterior["data_version"], "->", atual["data_version"])
    print("  features que entraram:", sorted(f_atu - f_ant) or "nenhuma")
    print("  features que saíram  :", sorted(f_ant - f_atu) or "nenhuma")
    print("  decisão:", atual["decisao"], "| vigente:", atual["vigente_apos"])

,cycle_id,data_version,lotes_novos,n_treino,n_validacao,features_eleitas,n_features,roc_auc_modelo_a,roc_auc_modelo_b,vencedor,roc_auc_vigente_anterior,decisao,vigente_apos,roc_auc_teste_vigente,drift_alerta
0,c20261123T181606Z-gh45236387855,v001-a59ff626,lote_001.csv,20715,5227,euribor3m|month|cons.conf.idx|nr.employed|emp....,10,0.6658,0.6479,logistic_regression,NaN,promover,c20261123T181606Z-gh45236387855__logistic_regr...,0.7164,False
1,c20261123T182253Z-gh45237612466,v002-86b526b7,lote_002.csv,22926,5796,euribor3m|month|cons.conf.idx|emp.var.rate|edu...,10,0.7008,0.7073,hist_gradient_boosting,0.6947,promover,c20261123T182253Z-gh45237612466__hist_gradient...,0.6440,True
2,c20261123T182934Z-gh45238828713,v003-48d4afe3,lote_003.csv,25129,6372,euribor3m|month|nr.employed|cons.conf.idx|emp....,10,0.6839,0.6887,hist_gradient_boosting,0.6890,manter_vigente,c20261123T182253Z-gh45237612466__hist_gradient...,0.6440,True
3,c20261123T183614Z-gh45240022843,v004-d9278016,lote_004.csv,27372,6910,euribor3m|emp.var.rate|cons.price.idx|contatad...,10,0.7364,0.7401,hist_gradient_boosting,0.6550,promover,c20261123T183614Z-gh45240022843__hist_gradient...,0.8110,True
4,c20261123T184307Z-gh45241187784,v005-6563af51,lote_005.csv,29617,7444,euribor3m|contatado_antes|emp.var.rate|cons.pr...,9,0.7990,0.8064,hist_gradient_boosting,0.7971,manter_vigente,c20261123T183614Z-gh45240022843__hist_gradient...,0.8110,True


O que mudou em relação ao ciclo anterior:
  data_version: v004-d9278016 -> v005-6563af51
  features que entraram: nenhuma
  features que saíram  : ['job']
  decisão: manter_vigente | vigente: c20261123T183614Z-gh45240022843__hist_gradient_boosting


## Teste integrado: o ciclo cumpriu todas as etapas?

In [5]:
configurar_mlflow(cfg)
runs = mlflow.search_runs(filter_string=f"tags.cycle_id = '{cycle_id}'", output_format="list")
etapas = sorted(r.data.tags.get("etapa") for r in runs)
verificacoes = {
    "votação refeita neste ciclo": (pasta_ciclo / "votacao.csv").exists() and "votacao_features" in etapas,
    "dois algoritmos treinados": etapas.count("treino_candidato") == 2,
    "decisão automática registrada": manifesto["ultimo_ciclo"]["cycle_id"] == cycle_id and "decisao_promocao" in etapas,
    "mesmas features nos dois candidatos": len({r.data.params.get("features") for r in runs if r.data.tags.get("etapa") == "treino_candidato"}) == 1,
    "previsões geradas": len(previsoes) == len(teste),
}
display(pd.Series(verificacoes, name="ok").to_frame())
assert all(verificacoes.values()), "Teste integrado reprovado"

with run_etapa(cfg, contexto, "previsoes"):
    mlflow.log_params({"vigente_cycle_id": manifesto["cycle_id"], "vigente_algoritmo": manifesto["algoritmo"], "vigente_run_id": manifesto["run_id"]})
    mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_teste.items() if isinstance(v, float)})
    mlflow.log_artifact(str(pasta_ciclo / "previsoes.csv"))
print("Ciclo", cycle_id, "concluído com sucesso.")

,ok
votação refeita neste ciclo,True
dois algoritmos treinados,True
decisão automática registrada,True
mesmas features nos dois candidatos,True
previsões geradas,True


Ciclo c20261123T184307Z-gh45241187784 concluído com sucesso.
